In [18]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [19]:
train_df = pd.read_csv("train_preprocessed.csv")

test_df = pd.read_csv("test_preprocessed.csv")

/tmp/ipykernel_3092/2367302725.py:1: DtypeWarning: Columns (43,51) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv("train_preprocessed.csv")
/tmp/ipykernel_3092/2367302725.py:3: DtypeWarning: Columns (43,51) have mixed types. Specify dtype option on import or set low_memory=False.
  test_df = pd.read_csv("test_preprocessed.csv")


In [20]:
city_cols = [col for col in train_df.columns if col.startswith("City_")]
county_cols = [col for col in train_df.columns if col.startswith("CountyOrParish_")]
district_cols = [col for col in train_df.columns if col.startswith("DistrictName_")]

features = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeArea",
    "Age",
    "Latitude",
    "Longitude"
] + city_cols + county_cols + district_cols

In [21]:
# Separate predictors and target variable for training and testing sets

X_train = train_df[features]
y_train = train_df['ClosePrice']
y_train_log = np.log1p(y_train)

X_test = test_df[features]
y_test = test_df['ClosePrice']

In [22]:
import xgboost as xgb
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

### XGBoost:


In [27]:
# Initialize XGBoost Regressor
xgb_model = xgb.XGBRegressor(random_state=42)

# Define a reduced hyperparameter grid for light tuning
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.1, 0.05]
}

In [29]:
# Initialize TimeSeriesSplit for cross-validation
tscv = TimeSeriesSplit(n_splits=3)

# Set up GridSearchCV with TimeSeriesSplit
grid_search = GridSearchCV(estimator=xgb_model, param_grid=param_grid,
                           cv=tscv, n_jobs=-1, verbose=2, scoring='neg_mean_squared_error')

# Fit the model
grid_search.fit(X_train, y_train_log)

Fitting 3 folds for each of 8 candidates, totalling 24 fits


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


GridSearchCV(cv=TimeSeriesSplit(gap=0, max_train_size=None, n_splits=3, test_size=None),
             estimator=XGBRegressor(base_score=None, booster=None,
                                    callbacks=None, colsample_bylevel=None,
                                    colsample_bynode=None,
                                    colsample_bytree=None, device=None,
                                    early_stopping_rounds=None,
                                    enable_categorical=True, eval_metric=None,
                                    feature_types=None, feature_weights=None,
                                    gamma=None,...
                                    max_cat_threshold=None,
                                    max_cat_to_onehot=None, max_delta_step=None,
                                    max_depth=None, max_leaves=None,
                                    min_child_weight=None, missing=nan,
                                    monotone_constraints=None,
                                    multi_strategy=None, n_estimators=None,
                                    n_jobs=None, num_parallel_tree=None, ...),
             n_jobs=-1,
             param_grid={'learning_rate': [0.1, 0.05], 'max_depth': [3, 5],
                         'n_estimators': [100, 200]},
             scoring='neg_mean_squared_error', verbose=2)

### Best parameters and score:

In [30]:
# Display the best parameters and best score
print("Best parameters found: ", grid_search.best_params_)
print("Best negative RMSE found: ", grid_search.best_score_)

# Get the best model
best_xgb_model = grid_search.best_estimator_

Best parameters found:  {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200}
Best negative RMSE found:  -0.06556867828748873


Make predictions on the test set using the best model and evaluate its performance:

In [31]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np

# Make predictions on the test set
y_pred_log = best_xgb_model.predict(X_test)
y_pred = np.expm1(y_pred_log) # Inverse transform to original scale

# Evaluate the model
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
mdape = np.median(np.abs((y_test - y_pred) / y_test)) * 100

print(f"RMSE on test set: {rmse}")
print(f"R-squared on test set: {r2}")
print(f"MAE on test set: {mae}")
print(f"MAPE on test set: {mape}%")
print(f"MdAPE on test set: {mdape}%")

RMSE on test set: 786071.0779840811
R-squared on test set: 0.739912860198685
MAE on test set: 251520.3954513256
MAPE on test set: 24.125065956232035%
MdAPE on test set: 11.687593201754385%
